In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 80)
plt.rcParams['figure.figsize'] = (9, 6)


# Si es necesario DIR_PROC = 'Data/processed'
# os.makedirs(DIR_PROC, exist_ok=True)

In [19]:
# Captura de version de Python y librerias clave para reproducibilidad --
# correr esto UNA VEZ en el entorno final antes de generar resultados definitivos,
# no en cualquier punto intermedio del desarrollo
import sys
import statsmodels
import sklearn
import pandas as pd

print(f"Python: {sys.version}")
print(f"statsmodels: {statsmodels.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"pandas: {pd.__version__}")

Python: 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
statsmodels: 0.14.6
scikit-learn: 1.9.0
pandas: 3.0.5


In [ ]:
df_unificado  = pd.read_csv('Data/processed/segmentos_unificado.csv')

In [12]:
df_unificado 

,long_seg_m,dist_puente_peatonal_m,dist_puente_vehicular_m,dist_semaforo_m,dist_peatonal_m,n_siniestros,poblacion_localidad_media,n_herido,n_muerto
0,30.958381,2149.708556,2248.410504,1410.063301,0.0,0,0.0,0,0
1,35.729601,2130.397897,2218.697332,1378.963885,0.0,0,0.0,0,0
2,31.900881,2108.777826,2173.881717,1232.553221,0.0,0,0.0,0,0
3,48.556662,2102.083822,2208.676848,1410.063301,0.0,0,0.0,0,0
4,21.075999,2106.174016,2164.688445,1255.390362,0.0,0,0.0,0,0
...,...,...,...,...,...,...,...,...,...
131699,138.327740,3199.597149,2074.655202,4829.450372,0.0,0,0.0,0,0
131700,47.720755,3310.416595,2202.181128,4938.129880,0.0,0,0.0,0,0
131701,131.444749,3349.305031,2246.587457,4976.154833,0.0,0,0.0,0,0
131702,204.070575,3349.305031,2246.587457,4976.154833,0.0,0,0.0,0,0


In [23]:
import statsmodels.api as sm
from sklearn.model_selection import train_test_split

# Semilla fija para reproducibilidad -- este numero debe quedar documentado
# textualmente en la metodologia, no solo en el codigo
SEMILLA = 1949

train, test = train_test_split(df_unificado, test_size=0.20, random_state=SEMILLA)
print(f"Entrenamiento: {len(train)} segmentos | Validacion: {len(test)} segmentos")

Entrenamiento: 105363 segmentos | Validacion: 26341 segmentos


In [33]:
import numpy as np

# OFFSET: aun no esta cerrado si combina longitud + poblacion o solo uno de los dos.
# Dejo la version combinada como placeholder -- AJUSTAR segun la decision final
train['offset'] = np.log(train['long_seg_m']) + np.log(train['poblacion_localidad_media'] + 1)
test['offset'] = np.log(test['long_seg_m']) + np.log(test['poblacion_localidad_media'] + 1)

covariables = ['dist_semaforo_m', 'dist_puente_peatonal_m', 'dist_puente_vehicular_m','dist_peatonal_m']

X_train = sm.add_constant(train[covariables])
X_test = sm.add_constant(test[covariables])
y_train = train['n_siniestros']
y_test = test['n_siniestros']

In [34]:
modelo_poisson = sm.GLM(
    y_train, X_train,
    family=sm.families.Poisson(),
    offset=train['offset']
).fit()

print(modelo_poisson.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:           n_siniestros   No. Observations:               105363
Model:                            GLM   Df Residuals:                   105358
Model Family:                 Poisson   Df Model:                            4
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -22759.
Date:                Mon, 28 Sep 2026   Deviance:                       21633.
Time:                        02:56:54   Pearson chi2:                 9.56e+04
No. Iterations:                    15   Pseudo R-squ. (CS):           0.001863
Covariance Type:            nonrobust                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                     

In [35]:
# sm.NegativeBinomial (NO sm.GLM) estima alpha por MLE junto a los coeficientes,
# con error estandar propio -- esto es lo que permite la prueba de razon de
# verosimilitud descrita en la metodologia; GLM con family NB NO sirve para esto
modelo_bn = sm.NegativeBinomial(
    y_train, X_train,
    offset=train['offset'],
    loglike_method='nb2'  # varianza = mu + alpha*mu^2, la parametrizacion estandar en literatura de siniestros
).fit(disp=0)

print(modelo_bn.summary())
alpha_estimado = modelo_bn.params['alpha']
print(f"Alpha estimado: {alpha_estimado:.4f}")

c:\Users\santiago\Desktop\Universidad\Tesis\Programacion\TT\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:3663: RuntimeWarning: overflow encountered in exp
  return np.exp(linpred)
c:\Users\santiago\Desktop\Universidad\Tesis\Programacion\TT\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:3379: RuntimeWarning: divide by zero encountered in log
  llf = coeff + size*np.log(prob) + endog*np.log(1-prob)
c:\Users\santiago\Desktop\Universidad\Tesis\Programacion\TT\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:3470: RuntimeWarning: overflow encountered in multiply
  dparams = exog*a1 * (y-mu)/(mu+a1)
c:\Users\santiago\Desktop\Universidad\Tesis\Programacion\TT\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:3470: RuntimeWarning: invalid value encountered in multiply
  dparams = exog*a1 * (y-mu)/(mu+a1)
c:\Users\santiago\Desktop\Universidad\Tesis\Programacion\TT\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:3470: Ru

                     NegativeBinomial Regression Results                      
Dep. Variable:           n_siniestros   No. Observations:               105363
Model:               NegativeBinomial   Df Residuals:                   105358
Method:                           MLE   Df Model:                            4
Date:                Mon, 28 Sep 2026   Pseudo R-squ.:                     nan
Time:                        02:59:33   Log-Likelihood:                -20843.
converged:                       True   LL-Null:                           nan
Covariance Type:            nonrobust   LLR p-value:                       nan
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                     -16.5809      0.022   -740.457      0.000     -16.625     -16.537
dist_semaforo_m            -0.0005   3.53e-05    -13.554      0.000      -0.001      -0.000


c:\Users\santiago\Desktop\Universidad\Tesis\Programacion\TT\.venv\Lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
c:\Users\santiago\Desktop\Universidad\Tesis\Programacion\TT\.venv\Lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


In [36]:
from scipy import stats

# Poisson es el caso limite de BN cuando alpha=0 -- por eso son modelos anidados
# y se puede aplicar LR test. PERO alpha=0 esta en la FRONTERA del espacio de
# parametros (alpha no puede ser negativo), por lo que la distribucion asintotica
# no es chi2(1) pura, sino una mezcla 50/50 de chi2(0) y chi2(1) (Cameron & Trivedi, 2013).
# El p-valor correcto es la MITAD del que arrojaria un chi2 test convencional.
LR_stat = 2 * (modelo_bn.llf - modelo_poisson.llf)
p_valor = 0.5 * (1 - stats.chi2.cdf(LR_stat, df=1))

print(f"Estadistico LR: {LR_stat:.4f}")
print(f"p-valor (corregido por frontera): {p_valor:.6f}")

if p_valor < 0.05:
    print("Alpha significativamente distinto de 0 -> se adopta Binomial Negativa")
    modelo_final = modelo_bn
else:
    print("No hay evidencia de sobredispersion significativa -> se conserva Poisson por parsimonia")
    modelo_final = modelo_poisson

Estadistico LR: 3831.2227
p-valor (corregido por frontera): 0.000000
Alpha significativamente distinto de 0 -> se adopta Binomial Negativa


In [37]:
print(f"AIC Poisson: {modelo_poisson.aic:.2f}")
print(f"AIC Binomial Negativa: {modelo_bn.aic:.2f}")

AIC Poisson: 45527.36
AIC Binomial Negativa: 41698.13


In [38]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Prediccion sobre el conjunto de validacion usando el offset de TEST, no de train
if modelo_final is modelo_bn:
    y_pred = modelo_final.predict(X_test, offset=test['offset'])
else:
    y_pred = modelo_final.predict(X_test, offset=test['offset'])

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAE (validacion): {mae:.4f}")
print(f"RMSE (validacion): {rmse:.4f}")

MAE (validacion): 0.1660
RMSE (validacion): 0.9878


In [39]:
# Genera el bloque de texto listo para pegar en la metodologia,
# con los valores reales en vez de placeholders -- revisar antes de copiar
reporte = f"""
N observaciones totales: {len(df_unificado)}
N entrenamiento: {len(train)} | N validacion: {len(test)}
Semilla: {SEMILLA}
Alpha BN estimado: {alpha_estimado:.4f}
LR statistic: {LR_stat:.4f} | p-valor: {p_valor:.6f}
AIC Poisson: {modelo_poisson.aic:.2f} | AIC BN: {modelo_bn.aic:.2f}
Modelo seleccionado: {'Binomial Negativa' if modelo_final is modelo_bn else 'Poisson'}
MAE validacion: {mae:.4f}
RMSE validacion: {rmse:.4f}
"""
print(reporte)


N observaciones totales: 131704
N entrenamiento: 105363 | N validacion: 26341
Semilla: 1949
Alpha BN estimado: 0.5217
LR statistic: 3831.2227 | p-valor: 0.000000
AIC Poisson: 45527.36 | AIC BN: 41698.13
Modelo seleccionado: Binomial Negativa
MAE validacion: 0.1660
RMSE validacion: 0.9878

